# 00: Barley seed data (download and extract)

Paper section: *Datasets and Compute Environment*. This notebook uses downloads and preprocesses the Barley Seed dataset, which is 3,121 X-ray CT scans of individual barley seeds from 28 seed types, grouped into 77 spikes, at 127 μm (Amézquita et al. 2022).

Source: Dryad, doi:[10.5061/dryad.rxwdbrv93](https://doi.org/10.5061/dryad.rxwdbrv93) (dataset version 6, CC0).

Note that only a small part of the archives is needed:

| archive | extracted | used by |
|---|---|---|
| `xray_ct_scans.tar.gz` (11 GB) | `xray_ct_scans/clean_founders_seeds/` (3,121 TIFFs, 155 MB) | 04a, 05a, 06 |
| `shape_descriptors.tar.gz` (166 MB) | `shape_descriptors/founder_seeds/combined_d158_T64.csv` (labels, spikes, 11 traditional descriptors, published ECT) | 04a, 04b, 06 |
| `dimension_reduction.tar.gz` (29 MB) | `dimension_reduction/umap/umap_158_16_50_0.1_12_manhattan_unsupervised.csv` (the published UMAP-12) | 04b |

If you already have the archives, set the environment variable `DRYAD_ARCHIVES` to the folder that holds them (or copy them to `data/raw/dryad/`) so you don't have to downlaod anything.

In [ ]:
import os
import tarfile
import urllib.request
from pathlib import Path

import barley as B

ARCHIVE_DIR = Path(os.environ.get("DRYAD_ARCHIVES", B.RAW / "dryad"))
DRYAD = "https://datadryad.org"
FILES = {   # archive -> (Dryad file download path, size in bytes)
    "xray_ct_scans.tar.gz": ("/api/v2/files/1115481/download", 11307781411),
    "shape_descriptors.tar.gz": ("/api/v2/files/1115484/download", 166231415),
    "dimension_reduction.tar.gz": ("/api/v2/files/1115457/download", 29152427),
}
EXTRACT = {  # archive -> (destination under data/raw, member prefix to extract)
    "xray_ct_scans.tar.gz": ("xray", "xray_ct_scans/clean_founders_seeds/"),
    "shape_descriptors.tar.gz": ("descriptors", "shape_descriptors/founder_seeds/combined_d158_T64.csv"),
    "dimension_reduction.tar.gz": ("dimred", "dimension_reduction/umap/umap_158_16_50_0.1_12_manhattan_unsupervised.csv"),
}
print("archives:", ARCHIVE_DIR)

## Download

In [ ]:
def download(name):
    path = ARCHIVE_DIR / name
    url, size = FILES[name]
    if path.exists() and path.stat().st_size == size:
        print(f"{name}: present")
        return
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    print(f"{name}: downloading {size / 2**30:.2f} GB from {DRYAD + url}", flush=True)
    with urllib.request.urlopen(DRYAD + url) as r, open(tmp, "wb") as f:
        done = 0
        while chunk := r.read(1 << 22):
            f.write(chunk)
            done += len(chunk)
            if done % (1 << 30) < (1 << 22):
                print(f"  {done / 2**30:.1f} GB", flush=True)
    assert tmp.stat().st_size == size, f"{name}: incomplete download"
    tmp.rename(path)


for name in FILES:
    download(name)

## Extract

Note that this can take a few minutes

In [ ]:
def extract(name):
    dest, prefix = EXTRACT[name]
    target = B.RAW / dest
    if (target / prefix.rstrip("/")).exists():
        print(f"{name}: already extracted")
        return
    n = 0
    with tarfile.open(ARCHIVE_DIR / name, "r|gz") as tar:
        for member in tar:
            if member.name.startswith(prefix):
                tar.extract(member, target, filter="data")
                n += member.isfile()
    print(f"{name}: extracted {n} files into {target.relative_to(B.ROOT)}")


for name in EXTRACT:
    extract(name)

## Check

In [ ]:
import numpy as np
import pandas as pd

d = B.load_table(with_files=True)
per_type = pd.Series(d["accession"]).value_counts()
print(f"{len(d['files'])} seeds, {len(per_type)} seed types ({per_type.min()}-{per_type.max()} seeds each), "
      f"{len(np.unique(d['panicle']))} spikes")
assert len(d["files"]) == 3121 and len(per_type) == 28 and len(np.unique(d["panicle"])) == 77
assert B.UMAP_FILE.exists()